# Band IoU Loss — 可微分双曲线带损失

现有参数损失（Smooth L1）把 width / height / thickness 当成三个独立数回归，
不感知最终 mask 的几何重叠质量。

本 notebook 实现：
1. **可微分软 mask 渲染器** — 用 sigmoid 代替 cv2.fillPoly，梯度可回传
2. **Band IoU Loss** — 直接优化预测带与 GT 带的几何重叠
3. **Band Dice Loss** — 备选，对正负样本不平衡更鲁棒
4. **可视化验证** — 软 mask vs 硬 mask 对比
5. **梯度验证** — 确认梯度能回传到参数
6. **训练集成** — 如何加入现有 train_one_epoch

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from typing import Tuple

## 1. 可微分软 mask 渲染器

对图像中每个像素 `(px, py)`，计算它到抛物线中心线的距离：

```
yc(x) = y_v + height * ((x - x_v) / half_w)²
dist   = |py - yc(px)|
```

用 sigmoid 做软阈值，得到可微分的 [0,1] mask：

```
band_value = sigmoid( (thickness/2 - dist) / temperature )
```

`temperature` 控制边界锐利程度：越小越接近硬 mask，越大越模糊。

In [ ]:
def soft_band_mask(
    h: int,
    w: int,
    x_v: torch.Tensor,       # 顶点 x，像素坐标
    y_v: torch.Tensor,       # 顶点 y，像素坐标
    width: torch.Tensor,     # 带的水平宽度，像素
    height: torch.Tensor,    # 顶点到边缘的纵向高度，像素
    thickness: torch.Tensor, # 带的厚度，像素
    temperature: float = 2.0,
) -> torch.Tensor:
    """
    返回 (H, W) 软 mask，值域 [0, 1]，全程可微分。
    """
    device = x_v.device
    ys = torch.arange(h, dtype=torch.float32, device=device)
    xs = torch.arange(w, dtype=torch.float32, device=device)
    py, px = torch.meshgrid(ys, xs, indexing='ij')  # (H, W)

    half_w = width / 2.0
    dx = (px - x_v) / (half_w + 1e-6)          # 归一化水平偏移
    yc = y_v + height * dx.pow(2)               # 抛物线中心线

    # 像素到中心线的距离
    dist = (py - yc).abs()

    # 纵向软阈值：距中心线 < thickness/2 则为带内
    band = torch.sigmoid((thickness / 2.0 - dist) / temperature)

    # 水平范围软阈值：x in [x_v - half_w, x_v + half_w]
    x_in = (
        torch.sigmoid((px - (x_v - half_w)) / temperature) *
        torch.sigmoid(((x_v + half_w) - px) / temperature)
    )

    return band * x_in  # (H, W)

## 2. Band IoU Loss 与 Band Dice Loss

In [ ]:
def band_iou_loss(
    h: int, w: int,
    pred_xv: torch.Tensor, pred_yv: torch.Tensor,
    pred_width: torch.Tensor, pred_height: torch.Tensor, pred_thickness: torch.Tensor,
    gt_xv: torch.Tensor, gt_yv: torch.Tensor,
    gt_width: torch.Tensor, gt_height: torch.Tensor, gt_thickness: torch.Tensor,
    temperature: float = 2.0,
) -> torch.Tensor:
    """
    1 - Soft IoU between predicted band and GT band.
    所有参数均为像素单位的标量 Tensor（支持梯度）。
    """
    pred_mask = soft_band_mask(h, w, pred_xv, pred_yv, pred_width, pred_height, pred_thickness, temperature)
    gt_mask   = soft_band_mask(h, w, gt_xv,   gt_yv,   gt_width,   gt_height,   gt_thickness,   temperature)

    inter = (pred_mask * gt_mask).sum()
    union = pred_mask.sum() + gt_mask.sum() - inter
    return 1.0 - inter / (union + 1e-6)


def band_dice_loss(
    h: int, w: int,
    pred_xv: torch.Tensor, pred_yv: torch.Tensor,
    pred_width: torch.Tensor, pred_height: torch.Tensor, pred_thickness: torch.Tensor,
    gt_xv: torch.Tensor, gt_yv: torch.Tensor,
    gt_width: torch.Tensor, gt_height: torch.Tensor, gt_thickness: torch.Tensor,
    temperature: float = 2.0,
) -> torch.Tensor:
    """
    1 - Soft Dice between predicted band and GT band.
    对正负不平衡更鲁棒（小目标场景推荐）。
    """
    pred_mask = soft_band_mask(h, w, pred_xv, pred_yv, pred_width, pred_height, pred_thickness, temperature)
    gt_mask   = soft_band_mask(h, w, gt_xv,   gt_yv,   gt_width,   gt_height,   gt_thickness,   temperature)

    inter = (pred_mask * gt_mask).sum()
    denom = pred_mask.sum() + gt_mask.sum()
    return 1.0 - 2.0 * inter / (denom + 1e-6)

## 3. 训练集成：masked_band_iou_loss

与现有 `peak_mask` 机制对接：只对有 GT 实例的峰值位置计算 Band Loss。

输入的参数 map 均为归一化值（来自模型输出和 GT），函数内部转换为像素单位。

In [ ]:
def masked_band_iou_loss(
    pred_param:  torch.Tensor,   # (B, 3, hm_h, hm_w)  归一化 [0,1]
    pred_offset: torch.Tensor,   # (B, 2, hm_h, hm_w)
    gt_param:    torch.Tensor,   # (B, 3, hm_h, hm_w)  归一化 [0,1]
    gt_offset:   torch.Tensor,   # (B, 2, hm_h, hm_w)
    peak_mask:   torch.Tensor,   # (B, 1, hm_h, hm_w)
    input_h: int, input_w: int,
    hm_h: int, hm_w: int,
    temperature: float = 2.0,
    loss_fn: str = 'iou',        # 'iou' 或 'dice'
) -> torch.Tensor:
    """
    在所有 GT 峰值位置计算 Band IoU / Dice Loss 并取平均。
    """
    B = pred_param.shape[0]
    losses = []

    for b in range(B):
        # 找到该样本所有 GT 峰值位置
        peak_yx = peak_mask[b, 0].nonzero(as_tuple=False)  # (N, 2) → (yi, xi)
        for yx in peak_yx:
            yi, xi = int(yx[0]), int(yx[1])

            # ── GT 参数（像素单位）──
            gt_dx = gt_offset[b, 0, yi, xi]
            gt_dy = gt_offset[b, 1, yi, xi]
            gt_xv = (xi + gt_dx) / hm_w * input_w
            gt_yv = (yi + gt_dy) / hm_h * input_h
            gt_w  = gt_param[b, 0, yi, xi] * input_w
            gt_h  = gt_param[b, 1, yi, xi] * input_h
            gt_t  = gt_param[b, 2, yi, xi] * input_h

            # ── 预测参数（像素单位）──
            pd_dx = pred_offset[b, 0, yi, xi]
            pd_dy = pred_offset[b, 1, yi, xi]
            pd_xv = (xi + pd_dx) / hm_w * input_w
            pd_yv = (yi + pd_dy) / hm_h * input_h
            pd_w  = pred_param[b, 0, yi, xi] * input_w
            pd_h  = pred_param[b, 1, yi, xi] * input_h
            pd_t  = pred_param[b, 2, yi, xi] * input_h

            if loss_fn == 'dice':
                loss = band_dice_loss(
                    input_h, input_w,
                    pd_xv, pd_yv, pd_w, pd_h, pd_t,
                    gt_xv, gt_yv, gt_w, gt_h, gt_t,
                    temperature,
                )
            else:
                loss = band_iou_loss(
                    input_h, input_w,
                    pd_xv, pd_yv, pd_w, pd_h, pd_t,
                    gt_xv, gt_yv, gt_w, gt_h, gt_t,
                    temperature,
                )
            losses.append(loss)

    if not losses:
        return pred_param.sum() * 0.0
    return torch.stack(losses).mean()

## 4. 可视化：软 mask vs 硬 mask

In [ ]:
def hard_band_mask(h, w, x_v, y_v, width, height, thickness):
    """现有代码的不可微硬 mask（用于对比）。"""
    width     = max(float(width),     2.0)
    height    = max(float(height),    1.0)
    thickness = max(float(thickness), 1.0)
    half_w    = width / 2.0
    n_pts     = max(40, int(round(width)))
    upper_pts, lower_pts = [], []
    for i in range(n_pts + 1):
        t  = i / max(n_pts, 1)
        x  = (x_v - half_w) + width * t
        dx = (x - x_v) / (half_w + 1e-6)
        yc = y_v + height * dx ** 2
        upper_pts.append((x, yc - thickness / 2.0))
        lower_pts.append((x, yc + thickness / 2.0))
    poly = np.array(upper_pts + list(reversed(lower_pts)), dtype=np.float32)
    poly[:, 0] = np.clip(poly[:, 0], 0, w - 1)
    poly[:, 1] = np.clip(poly[:, 1], 0, h - 1)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [np.round(poly).astype(np.int32)], 1)
    return mask.astype(np.float32)

In [ ]:
H, W = 224, 224

# GT 参数
params_gt = dict(x_v=112.0, y_v=50.0, width=160.0, height=80.0, thickness=10.0)

# 预测参数（故意偏移，模拟误差）
params_pred = dict(x_v=118.0, y_v=55.0, width=150.0, height=70.0, thickness=12.0)

def to_t(v, requires_grad=False):
    return torch.tensor(v, dtype=torch.float32, requires_grad=requires_grad)

# 软 mask
soft_gt   = soft_band_mask(H, W, to_t(params_gt['x_v']),   to_t(params_gt['y_v']),
                           to_t(params_gt['width']),   to_t(params_gt['height']),   to_t(params_gt['thickness']),   temperature=2.0)
soft_pred = soft_band_mask(H, W, to_t(params_pred['x_v']), to_t(params_pred['y_v']),
                           to_t(params_pred['width']), to_t(params_pred['height']), to_t(params_pred['thickness']), temperature=2.0)

# 硬 mask
hard_gt   = hard_band_mask(H, W, **params_gt)
hard_pred = hard_band_mask(H, W, **params_pred)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
titles = [
    ['Soft GT', 'Soft Pred', 'Soft 差异 (|GT - Pred|)'],
    ['Hard GT', 'Hard Pred', 'Hard 差异 (|GT - Pred|)'],
]
soft_data = [soft_gt.detach().numpy(), soft_pred.detach().numpy(),
             (soft_gt - soft_pred).abs().detach().numpy()]
hard_data = [hard_gt, hard_pred, np.abs(hard_gt - hard_pred)]

for row, (data_list, title_list) in enumerate(zip([soft_data, hard_data], titles)):
    for col, (data, title) in enumerate(zip(data_list, title_list)):
        ax = axes[row][col]
        im = ax.imshow(data, cmap='hot', vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('软 mask vs 硬 mask 对比  (temperature=2.0)', fontsize=13)
plt.tight_layout()
plt.show()

# 打印 IoU 值
inter_s = (soft_gt * soft_pred).sum().item()
union_s = (soft_gt.sum() + soft_pred.sum() - inter_s)
print(f'Soft IoU  = {inter_s / (union_s + 1e-6):.4f}')

inter_h = float((hard_gt * hard_pred).sum())
union_h = float(hard_gt.sum() + hard_pred.sum() - inter_h)
print(f'Hard IoU  = {inter_h / (union_h + 1e-6):.4f}')

## 5. Temperature 对软 mask 的影响

In [ ]:
temps = [0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(1, len(temps), figsize=(14, 4))

for ax, temp in zip(axes, temps):
    m = soft_band_mask(H, W, to_t(112.0), to_t(50.0),
                       to_t(160.0), to_t(80.0), to_t(10.0), temperature=temp)
    ax.imshow(m.detach().numpy(), cmap='hot', vmin=0, vmax=1)
    ax.set_title(f'temperature={temp}')
    ax.axis('off')

plt.suptitle('Temperature 越小越接近硬 mask', fontsize=12)
plt.tight_layout()
plt.show()

## 6. 梯度验证

确认 Band IoU Loss 的梯度能正确回传到各参数。

In [ ]:
# 创建带梯度的参数
pd_xv = to_t(118.0, requires_grad=True)
pd_yv = to_t(55.0,  requires_grad=True)
pd_w  = to_t(150.0, requires_grad=True)
pd_h  = to_t(70.0,  requires_grad=True)
pd_t  = to_t(12.0,  requires_grad=True)

gt_xv = to_t(112.0)
gt_yv = to_t(50.0)
gt_w  = to_t(160.0)
gt_h  = to_t(80.0)
gt_t  = to_t(10.0)

loss = band_iou_loss(
    H, W,
    pd_xv, pd_yv, pd_w, pd_h, pd_t,
    gt_xv, gt_yv, gt_w, gt_h, gt_t,
    temperature=2.0,
)

loss.backward()

print(f'Band IoU Loss = {loss.item():.4f}  (越小越好)')
print()
print('梯度（非 None 说明可回传）：')
for name, param in [('x_v', pd_xv), ('y_v', pd_yv), ('width', pd_w),
                    ('height', pd_h), ('thickness', pd_t)]:
    grad = param.grad
    print(f'  d(loss)/d({name:10s}) = {grad.item():+.6f}')

## 7. 加入训练循环

在现有 `0607-1.ipynb` 的 `train_one_epoch` 中，总损失加入 Band IoU Loss：

```python
LAM_BAND = 0.5   # 新超参，建议从 0.1 开始调

loss = (
    focal_loss_heatmap(pred_hm, gt_hm, peak_mask)
    + LAM * masked_param_loss_spatial(pred_param, gt_param, peak_mask)
    + masked_offset_loss(pred_offset, gt_offset, peak_mask)
    + LAM_BAND * masked_band_iou_loss(
        pred_param, pred_offset,
        gt_param,   gt_offset,
        peak_mask,
        input_h=input_size[0], input_w=input_size[1],
        hm_h=input_size[0] // HM_STRIDE,
        hm_w=input_size[1] // HM_STRIDE,
        temperature=2.0,
        loss_fn='iou',
    )
)
```

### 注意事项
- `masked_band_iou_loss` 内部有 Python 循环，每 batch 耗时略增（通常 <20%）
- `temperature` 建议训练前期用较大值（5.0），后期退火到小值（1.0）
- 若样本中无 GT 峰值，函数返回 0，不影响训练稳定性

In [ ]:
# ── 模拟一个 batch 的前向+损失计算，验证 masked_band_iou_loss 可正常运行 ──

B, hm_h, hm_w = 2, 28, 28   # 224 / 8
input_h, input_w = 224, 224

torch.manual_seed(0)
pred_param  = torch.sigmoid(torch.randn(B, 3, hm_h, hm_w, requires_grad=True))
pred_offset = torch.tanh(torch.randn(B, 2, hm_h, hm_w)) * 0.5
gt_param    = torch.sigmoid(torch.randn(B, 3, hm_h, hm_w))
gt_offset   = torch.zeros(B, 2, hm_h, hm_w)

# 每个 batch 随机设置 1-2 个峰值
peak_mask = torch.zeros(B, 1, hm_h, hm_w)
peak_mask[0, 0, 10, 14] = 1.0
peak_mask[1, 0, 8,  12] = 1.0
peak_mask[1, 0, 18, 20] = 1.0

loss = masked_band_iou_loss(
    pred_param, pred_offset,
    gt_param,   gt_offset,
    peak_mask,
    input_h=input_h, input_w=input_w,
    hm_h=hm_h, hm_w=hm_w,
    temperature=2.0,
    loss_fn='iou',
)

loss.backward()

print(f'masked_band_iou_loss = {loss.item():.4f}')
print(f'pred_param.grad is not None: {pred_param.grad is not None}')
print('✓ 梯度正常回传')